# Week 7 capstone: Step 5, noise pilot (arm A, seeds 901 and 902)

**Scope: the two-run noise pilot only.** It estimates how much the reference arm's validation MAPE varies from seed to seed. It does **not** compare positional encodings, set δ or k, freeze the pre-registration, or run any main experiment, and the repository refuses those until their gates are met.

- **What runs.** Arm A (learned absolute) only, seeds 901 and 902, 8000 steps, policy P1, the existing dataset. Two runs, about 5 minutes each. Seed 900 at 8000 steps is reused from calibration (the summary checks that its configuration is identical).
- **What is reported.** Per-seed validation MAPE, mean, standard deviation (n-1), range, an interval for the true SD (wide, with only three seeds), GPU time, memory, environment and provenance. Test-set metrics are never read.
- **No `pip install`, no upgrades.** The environment must match `requirements.lock.txt` or the run is refused.

**Provenance.** The commit hash is already filled in cell 1, so there is nothing to edit. The cell checks out exactly that commit and stops if it cannot. It prints whether the code came from an uploaded zip or from GitHub. Uploading the zip is optional.

Run the cells top to bottom on a **T4 GPU runtime**, then download the zip from the last cell and send it back.

In [ ]:
# 1. Get the code, from week7_repo.zip if you uploaded it to the Files panel, otherwise from GitHub.
#    COMMIT is already filled in, so there is nothing to paste. The cell stops if the code is on a different commit.
REPO_URL = "https://github.com/sweiss1367/wk5_architecture-variants.git"
BRANCH = "claude/trusting-pascal-1qjsp0"
COMMIT = "03d5e16d61b548440e6eb691913f51fd126c4f11"   # pre-filled: the exact code the study runs on. Nothing to edit. The cell stops if it cannot get this commit

import os, subprocess, time
if os.path.isdir("repo"):                  # never reuse a stale copy: keep the old one aside
    os.rename("repo", f"repo_old_{int(time.time())}")
used_zip = os.path.exists("week7_repo.zip")
if used_zip:
    subprocess.run(["unzip", "-q", "week7_repo.zip"], check=True)      # creates ./repo, including its .git history
else:
    url = REPO_URL
    try:                                   # private repo: add a Colab secret named GITHUB_TOKEN (never printed)
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "remote", "set-url", "origin", REPO_URL], check=True)   # drop the token from the remote
if COMMIT:
    subprocess.run(["git", "-C", "repo", "checkout", COMMIT], check=True)
head = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code source:", "uploaded zip" if used_zip else "GitHub clone", "| HEAD:", head)
if not COMMIT:
    raise SystemExit("COMMIT is empty. Paste the commit hash from the instructions, then run this cell again.")
if not head.startswith(COMMIT):
    raise SystemExit(f"Wrong commit: HEAD is {head}, expected {COMMIT}. Stopping so the provenance record stays exact.")
%cd repo/week7_capstone
!git status --short

In [ ]:
# 2. GPU check (all earlier sessions used a Tesla T4; a different GPU is allowed but changes absolute timings).
!nvidia-smi --query-gpu=name,uuid,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU and restart."
name = torch.cuda.get_device_name(0)
print(torch.__version__, torch.version.cuda, name)
if "T4" not in name:
    print("WARNING: not a T4. The pilot will still run, but tell the reviewer.")

In [ ]:
# 3. Gates, checked before any run: environment equals the lock, code equals the commit, the policy decision agrees with the
#    config, and the step count is the approved 8000. (run_ablation.py checks the gates again for every run.)
import glob, json, subprocess, yaml
subprocess.run(["python", "src/check_env.py", "--verify"], check=True)
subprocess.run(["git", "diff", "--quiet", "--", ".", ":(exclude)results", ":(exclude)data"], check=True)   # code and configs only; runs write to results/
cfg = yaml.safe_load(open("configs/base.yaml"))
decision = json.load(open("results/determinism_decision.json"))["selected_policy"]
print("policy decision:", decision, "| config policy:", cfg["determinism"]["policy"], "| train.steps:", cfg["train"]["steps"])
assert decision == cfg["determinism"]["policy"] == "P1" and cfg["train"]["steps"] == 8000
assert glob.glob("results/runs/calibration/A_seed900_S8000_attempt1.json"), "seed 900 at 8000 steps (reused as a pilot seed) is missing"
print("existing runs:", {s: len(glob.glob(f"results/runs/{s}/*.json")) for s in ("calibration", "pilot", "main", "extension")})

In [ ]:
# 4. The noise pilot: arm A, seeds 901 and 902, then the seed-level summary.
#    Safe to re-run: finished runs are skipped, so it resumes where it stopped.
#    Each run takes about 5 minutes and prints one line when it finishes. Do NOT interrupt the cell while a run is going.
import glob, os, subprocess, sys

def status():
    done = sorted(os.path.basename(f) for f in glob.glob("results/runs/pilot/A_seed90[12]_attempt*.json"))
    marks = sorted(os.path.basename(f) for f in glob.glob("results/runs/pilot/*.running"))
    print("pilot runs finished:", done or "none", "| unfinished markers:", marks or "none")

status()
proc = subprocess.Popen([sys.executable, "-u", "src/run_pilot.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:                 # stream the driver's output into the cell as it happens
    print(line, end="")
proc.wait()
status()
print("exit code:", proc.returncode)
assert proc.returncode == 0 and os.path.exists("results/pilot_summary.json"), \
    "The pilot did not finish. Paste this cell's whole output."

In [ ]:
# 5. Package the pilot outputs (kept apart from calibration and main results) and download them.
import glob, os, subprocess
paths = ["results/runs/pilot", "results/pilot_summary.json", "results/env_report.json", "results/session_manifest.json"] + glob.glob("results/pip_freeze_*.txt")
paths = [p for p in paths if os.path.exists(p)]
subprocess.run(["zip", "-rq", "week7_pilot_outputs.zip", *paths], check=True)
print("zipped:", paths)
from google.colab import files
files.download("week7_pilot_outputs.zip")

**Stop here.** Send back `week7_pilot_outputs.zip`. The next steps (review of the observed MAPE values and seed-level standard deviation, the proposed δ and the exact decision rule, then your approval before the pre-registration is frozen) happen only after the results are reviewed.